## Setup


In [ ]:
%pip install -q transformers==4.48.1 "tokenizers>=0.21,<0.22" "huggingface_hub>=0.26,<1.0" peft librosa accelerate gTTS


In [ ]:
import base64
import json
import re
import subprocess
import time
from pathlib import Path

import librosa
import numpy as np
import torch
import transformers
from google.colab import output as colab_output, userdata
from gtts import gTTS
from huggingface_hub import login, whoami
from IPython.display import Audio, Javascript, display

try:
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    login()

print("Logged in as", whoami()["name"])


In [ ]:
MODEL_ID = "fixie-ai/ultravox-v0_5-llama-3_1-8b"

pipe = transformers.pipeline(
    model=MODEL_ID,
    trust_remote_code=True,
    device_map="auto",
    torch_dtype=torch.float16,
)


In [ ]:
SYSTEM_PROMPT = '''
You are a  virtual Proactive Care assistant for a food-delivery platform.
You hear the captain's reply as audio. Detect the language the captain just spoke before answering: urdu, english, or arabic.
Reply in that language, (if the langauge switches between turns use the latest language spoken or the most predominant and likely candidate) and keep the entire answer in one language.

Order context:

- Captain (Driver): Ahmed K. Order: 16378920. Restaurant: Nando's JBR.
- The captain has been assigned but pickup is not confirmed. 
- The order is already late.


- Use only these facts. What was ordered is not proof of what was collected.
- Ask one focused question at a time. If the captain already gave an ETA or reason, use it rather than asking again.
- If there is a safety issue, stop routine questions and say a human agent will call them back.
- Never claim you have already cancelled, reassigned, or applied a penalty.
- Supported Languages: 
    - Urdu, 
    - English
    - EgyptianArabic.
    - Write Urdu replies in Urdu script, Egyptian Arabic in conversational Cairo Arabic, and English in clear simple English.
- Give only the last four digits of the order once, and never read the full order ID.
- Keep replies short, polite, and natural.

Output only JSON: {"detected_language": "<urdu|english|arabic>", "reply": "<spoken reply>", "next_action": "<one of: ask_order_status, ask_eta_minutes, ask_clarification, acknowledge_wait, explain_decline_penalty, escalate_to_support, safety_callback, decline_out_of_scope, switch_language, close_call>", "reason": "<short English reason>"}
'''

def load_audio(path, sr=16000):
    audio, _ = librosa.load(path, sr=sr, mono=True)
    return audio.astype(np.float32), sr

def output_text(output):
    if isinstance(output, list):
        output = output[0] if output else ''
    if isinstance(output, dict):
        output = output.get('generated_text', output.get('text', str(output)))
    return output if isinstance(output, str) else str(output)

def parse_json_out(text):
    text = output_text(text)
    match = re.search(r'\{.*\}', text, re.S)
    if not match:
        return {'detected_language': None, 'reply': text.strip(), 'next_action': 'ask_clarification', 'reason': 'No JSON found'}
    try:
        out = json.loads(match.group(0))
    except Exception:
        out = {'detected_language': None, 'reply': text.strip(), 'next_action': 'ask_clarification', 'reason': 'JSON parse failed'}
    out.setdefault('detected_language', None)
    out.setdefault('reply', '')
    out.setdefault('next_action', 'ask_clarification')
    out.setdefault('reason', 'Missing reason')
    return out

def run_turn(audio_path, max_new_tokens=80):
    audio, sr = load_audio(audio_path)
    output = pipe({'audio': audio, 'turns': [{'role': 'system', 'content': SYSTEM_PROMPT}], 'sampling_rate': sr}, max_new_tokens=max_new_tokens)
    return parse_json_out(output)

print('Ready: use run_turn(path_to_audio)')


## Evaluate


In [ ]:
def evaluate_recording(audio_path, captain_language, expected_action=None):
    started = time.time()
    result = run_turn(audio_path)
    reply = result.get("reply", "")
    checks = {
        "language_detected": result.get("detected_language") == captain_language,
        "valid_action": result.get("next_action") in {"ask_order_status", "ask_eta_minutes", "ask_clarification", "acknowledge_wait", "explain_decline_penalty", "escalate_to_support", "safety_callback", "decline_out_of_scope", "switch_language", "close_call"},
        "no_digits_in_reply": not re.search(r"\d", reply),
        "full_order_id_not_repeated": "16378920" not in reply.replace(" ", ""),
        "one_question_at_most": reply.count("?") + reply.count("؟") <= 1,
        "latency_under_two_seconds": time.time() - started <= 2,
    }
    if expected_action is not None:
        checks["expected_action"] = result.get("next_action") == expected_action
    result["latency_seconds"] = round(time.time() - started, 2)
    result["checks"] = checks
    print(json.dumps(result, ensure_ascii=False, indent=2))
    return result

test_audio_path = Path("calls/example_call/turn_01.wav")
if test_audio_path.exists():
    test_result = evaluate_recording(test_audio_path, captain_language="urdu")
else:
    print("Add a recording at", test_audio_path, "or set test_audio_path to a local audio file.")


## Live


In [ ]:
TTS_LANGUAGE_CODES = {"urdu": "ur", "english": "en", "arabic": "ar"}
RECORDER_JS = """
async function recordAudio(ms) {
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const recorder = new MediaRecorder(stream);
  const chunks = [];
  recorder.ondataavailable = event => chunks.push(event.data);
  const stopped = new Promise(resolve => recorder.onstop = resolve);
  recorder.start();
  await new Promise(resolve => setTimeout(resolve, ms));
  recorder.stop();
  await stopped;
  stream.getTracks().forEach(track => track.stop());
  const bytes = new Uint8Array(await new Blob(chunks).arrayBuffer());
  let binary = "";
  for (let i = 0; i < bytes.length; i += 32768) {
    binary += String.fromCharCode.apply(null, bytes.subarray(i, i + 32768));
  }
  return btoa(binary);
}
"""


TTS_LANGUAGE_CODES = {"urdu": "ur", "english": "en", "arabic": "ar"}
RECORDER_JS = """
async function recordAudio(ms) {
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const recorder = new MediaRecorder(stream);
  const chunks = [];
  recorder.ondataavailable = event => chunks.push(event.data);
  const stopped = new Promise(resolve => recorder.onstop = resolve);
  recorder.start();
  await new Promise(resolve => setTimeout(resolve, ms));
  recorder.stop();
  await stopped;
  stream.getTracks().forEach(track => track.stop());
  const bytes = new Uint8Array(await new Blob(chunks).arrayBuffer());
  let binary = "";
  for (let i = 0; i < bytes.length; i += 32768) {
    binary += String.fromCharCode.apply(null, bytes.subarray(i, i + 32768));
  }
  return btoa(binary);
}
"""


def speak(text, language="english"):
    gTTS(text=text, lang=TTS_LANGUAGE_CODES.get(language, "en")).save("assistant_reply.mp3")
    display(Audio("assistant_reply.mp3", autoplay=True))


def live_turn(seconds=5, sample_rate=16000):
    print(f"Recording for {seconds} seconds. Allow microphone access in the browser.")
    display(Javascript(RECORDER_JS))
    encoded = colab_output.eval_js(f"recordAudio({int(seconds * 1000)})")
    Path("captain_turn.webm").write_bytes(base64.b64decode(encoded))
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", "captain_turn.webm", "-ac", "1", "-ar", str(sample_rate), "captain_turn.wav"], check=True)
    return load_audio("captain_turn.wav", sample_rate)


TRANSCRIBE_PROMPT = "Transcribe the captain audio verbatim in its original language and script. Output only the transcript."
LIVE_HISTORY = []


def run_live_audio(audio, sample_rate=16000, max_new_tokens=100):
    transcript_output = pipe(
        {"audio": audio, "turns": [{"role": "system", "content": TRANSCRIBE_PROMPT}], "sampling_rate": sample_rate},
        max_new_tokens=120,
    )
    transcript = output_text(transcript_output)
    turns = [{"role": "system", "content": SYSTEM_PROMPT}, *LIVE_HISTORY]
    output = pipe(
        {"audio": audio, "turns": turns, "sampling_rate": sample_rate},
        max_new_tokens=max_new_tokens,
    )
    result = parse_json_out(output)
    result["captain_transcript"] = transcript.strip()
    LIVE_HISTORY.extend([
        {"role": "user", "content": transcript.strip() or "[audio turn]"},
        {"role": "assistant", "content": result.get("reply", "")},
    ])
    return result


def live_conversation(seconds=5, language="urdu"):
    LIVE_HISTORY.clear()
    greetings = {
        "urdu": "السلام علیکم کیپٹن احمد، میں Nando's JBR کے order کے بارے میں call کر رہی ہوں۔ کیا آپ بات کر رہے ہیں؟",
        "english": "Hello Captain Ahmed, I'm calling about the order from Nando's JBR. Is this a good time to talk?",
        "arabic": "أهلاً يا كابتن أحمد. بكلمك بخصوص الـ order من Nando's JBR. ينفع أتكلم معاك؟",
    }
    greeting = greetings[language]
    print("Assistant:", greeting)
    LIVE_HISTORY.append({"role": "assistant", "content": greeting})
    speak(greeting, language)
    while True:
        if input("Press Enter to record, or type q to stop: ").strip().lower() == "q":
            break
        audio, sample_rate = live_turn(seconds=seconds)
        started = time.time()
        result = run_live_audio(audio, sample_rate)
        result["latency_seconds"] = round(time.time() - started, 2)
        print("Captain:", result.get("captain_transcript", ""))
        print("Assistant:", result.get("reply", ""))
        print("Detected language/action:", result.get("detected_language"), "/", result.get("next_action"))
        if result.get("reply"):
            speak(result["reply"], result.get("detected_language") or language)
        if result.get("next_action") == "close_call":
            break


live_conversation(seconds=30, language="urdu")
